In [ ]:
import numpy as np
import sep
from astropy.io import fits
import matplotlib.pyplot as plt
from matplotlib import rcParams
from astropy.table import Table
from matplotlib.patches import Ellipse

%matplotlib inline

rcParams['figure.figsize'] = [10., 8.]

In [ ]:
# input parameters

input_file = "./input.fits" # http://odahub.io/ontology#POSIXPath
threshold = 1.5 # http://odahub.io/ontology#Float

In [ ]:
hdul = fits.open(input_file)
data = hdul[0].data
data = data.astype(data.dtype.newbyteorder('='))

In [ ]:
# measure a spatially varying background on the image
bkg = sep.Background(data)

In [ ]:
# get a "global" mean and noise of the image background:
print(bkg.globalback)
print(bkg.globalrms)

In [ ]:
# evaluate background as 2-d array, same size as original image
bkg_image = bkg.back()

# evaluate the background noise as 2-d array, same size as original image
bkg_rms = bkg.rms()

# subtract the background
data_sub = data - bkg

In [ ]:
objects = sep.extract(data_sub, threshold, err=bkg.globalrms)

# how many objects were detected
len(objects)

In [ ]:
# show the background
fig, ax = plt.subplots()
im = ax.imshow(bkg_image, interpolation='nearest', cmap='gray', origin='lower')
fig.colorbar(im)
fig.savefig('bkg_image.png',format='png',bbox_inches='tight')

# show the background noise
fig, ax = plt.subplots()
im = ax.imshow(bkg_rms, interpolation='nearest', cmap='gray', origin='lower')
fig.colorbar(im)
fig.savefig('bkg_rms.png',format='png',bbox_inches='tight')

# plot background-subtracted image
fig, ax = plt.subplots()
m, s = np.mean(data_sub), np.std(data_sub)
im = ax.imshow(data_sub, interpolation='nearest', cmap='gray',
               vmin=m-s, vmax=m+s, origin='lower')

# plot an ellipse for each object
for i in range(len(objects)):
    e = Ellipse(xy=(objects['x'][i], objects['y'][i]),
                width=6*objects['a'][i],
                height=6*objects['b'][i],
                angle=objects['theta'][i] * 180. / np.pi)
    e.set_facecolor('none')
    e.set_edgecolor('red')
    ax.add_artist(e)

fig.savefig('sources.png',format='png',bbox_inches='tight')

plt.show()

In [ ]:
from oda_api.data_products import PictureProduct, ODAAstropyTable

bin_bkg_image = PictureProduct.from_file('bkg_image.png')
bin_rms_image = PictureProduct.from_file('bkg_rms.png')
bin_sor_image = PictureProduct.from_file('sources.png')

cat =  ODAAstropyTable(Table(data=objects))

In [ ]:
picture1 = bin_bkg_image # http://odahub.io/ontology#ODAPictureProduct
picture2 = bin_rms_image # http://odahub.io/ontology#ODAPictureProduct
picture3 = bin_sor_image # http://odahub.io/ontology#ODAPictureProduct
catalog_table = cat # http://odahub.io/ontology#ODAAstropyTable